In [ ]:
# 图片输入须固定到已交付 image_uri 的版本；旧公共表需先显式迁移/目录发布，再填实际版本。
# 第一格：粗筛通过概念及 taxonomy → 固定种子选 5 个 → 关联已审核图文 → 独立 Codex 出题。
# 手动执行本格会启动 5 次独立 Codex exec；已有输出为历史记录，运行后刷新。
import asyncio
import os
import sys
from pathlib import Path
from importlib import reload

import pandas as pd
from IPython.display import display

PROJECT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
DATA_ROOT = PROJECT.parent
for path in (PROJECT, DATA_ROOT / 'demiflow'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
os.environ['DEMIWTG_DATASETS_ROOT'] = str(DATA_ROOT)
# 兼容已运行过旧入口的长驻 kernel：保留原 notebook 对 reader、模型客户端及业务模块的刷新。
from demiflow.data import api as data_api
reload(data_api)
from demiflow.data import read_api
reload(read_api)
from demiflow import data
reload(data)
from demiflow.operator_llm import client as model_client
reload(model_client)
from benchmark.t2i.v2.operaters import concept_context as context_ops, images as image_ops, authoring as authoring_ops
reload(context_ops)
reload(image_ops)
reload(authoring_ops)
from benchmark.t2i.v2 import t2i_v2_benchmark_pipeline as pipeline
reload(pipeline)

# 1. 固定粗筛目标：1000 个概念中 469 个 screened/keep，471 hold、60 failed 不进入本批。
# 顶层 keep 已是两个模型有效且全部 keep；传名称与完整 taxonomy，不传粗筛理由/候选考点。
SCREENING_SOURCE = {
    'uri': 'demiwtg/benchmark/t2i/fine_screening/datasets/results__high_l3_categories_uniform1000_v1_20260927.lance',
    'version': 1,
}
SAMPLE_SIZE = 5
SAMPLE_SEED = 20260928  # 对全部通过概念按 seed/name 哈希选样，重复执行名单相同；不足 5 个报错。
# 恢复原 notebook 的固定 preparation 来源；只接 reviewed 文章与 published/keep 图片。
# 已只读核对：本次同样的 5 个概念在这两个版本均为 0 条可用图文；来源接通不代表每概念都有材料。
# 有材料时传完整正文与真实图片；无材料仍可出题，必要时检索核验。图片来源也可显式改为其他已核对版本。
ARTICLE_SOURCE = {'uri': 'demiwtg/preparation/articles/datasets/articles.lance', 'version': 8}
VISUAL_SOURCE = {'uri': 'demiwtg/preparation/datasets/image_catalog__object_migration_public_v5.lance', 'version': 1}
MAX_REFERENCE_IMAGES = 8  # 每概念最多提供 8 张可用原图；不足取实际数量，不借用其他概念图片。
MAX_CONTEXT_CHARS = 60000  # 完整 taxonomy、审定资料与原文、作答材料及 prompt 超限时跳过，不截断材料。
DOCUMENT_READS = False  # 按需补读审定文档；启用时使用 offline/local/modelhub、新 run/目标及已采纳来源。
DOCUMENT_READ_MAX_TURNS = 4  # 含最终回答；max_calls 的总额度需覆盖预期轮数。
DOCUMENT_READ_MAX_CALLS_PER_TURN = 2
DOCUMENT_READ_MAX_CHARS = 12000  # 单次完整材料，非模型 token 上限。
DOCUMENT_READ_MAX_BYTES = 8 * 1024 * 1024  # 单份 normalized 文档读取前字节上限。
DOCUMENT_READ_TIMEOUT_S = 30

# 2. 每概念新建独立 CLI 会话：临时目录、ephemeral、忽略用户 config、项目文档预算为零。
# 不 resume，不继承这次讨论或其他概念的历史；Codex 基础指令仍存在，不宣称空白系统上下文。
MODE = 'codex'
MODEL = 'gpt-6-astra'  # 本机模型缓存中可见；实际服务可用性由手工试跑验证。
REASONING_EFFORT = 'xhigh'  # Extra high（极高）；按当前调用参数传给 Codex。
CODEX_BIN = '/root/.vscode-server-codex/.vscode-server/extensions/openai.chatgpt-26.5917.62051/bin/linux-x86_64/codex'
CODEX_WEB_SEARCH = 'live'  # 允许按需查事实，不强制每题检索；实际来源写在 basis。
CONCURRENCY = 1
MAX_CALLS = SAMPLE_SIZE  # 最多 5 次新 exec；每次 exec 内可能有多轮模型/工具调用。
TIMEOUT_S = 900  # 每个概念最多等待 15 分钟；Codex 不采用 HTTP max_output_tokens/temperature。

# 3. 本批使用新 run/目标，保留历史题表。prompt/输出结构、规模、种子或模型改变时换新 RUN_ID 留对照。
RUN_ID = 'dual_keep_codex5_criteria_v3_20260928'  # 新增 criterion 字段，使用新目标表；旧批次及保存输出保留。
OUTPUT_TABLE_URI = f'demiwtg/benchmark/t2i/v2/datasets/candidates__{RUN_ID}.lance'
WRITE_MODE = 'overwrite'
RUN_DIR = DATA_ROOT / pipeline.DATASETS / RUN_ID
CONCEPT_AUDIT_SOURCE = None  # 可改为 preparation/concepts 的已采纳 uri/version；启用时改新 RUN_ID/目标。
CONFIG = pipeline.config(
    run=RUN_DIR, screening_source=SCREENING_SOURCE if CONCEPT_AUDIT_SOURCE is None else None,
    concept_audit_source=CONCEPT_AUDIT_SOURCE,
    sample_size=SAMPLE_SIZE, sample_seed=SAMPLE_SEED,
    article_source=ARTICLE_SOURCE, visual_source=VISUAL_SOURCE,
    target_uri=OUTPUT_TABLE_URI, write_mode=WRITE_MODE,
    mode=MODE, model=MODEL, codex_bin=CODEX_BIN, reasoning_effort=REASONING_EFFORT,
    codex_web_search=CODEX_WEB_SEARCH, concurrency=CONCURRENCY, queue_depth=1,
    max_calls=MAX_CALLS, timeout_s=TIMEOUT_S,
    max_reference_images=MAX_REFERENCE_IMAGES, max_context_chars=MAX_CONTEXT_CHARS,
    document_reads=DOCUMENT_READS, document_read_max_turns=DOCUMENT_READ_MAX_TURNS,
    document_read_max_calls_per_turn=DOCUMENT_READ_MAX_CALLS_PER_TURN,
    document_read_max_chars=DOCUMENT_READ_MAX_CHARS, document_read_max_bytes=DOCUMENT_READ_MAX_BYTES,
    document_read_timeout_s=DOCUMENT_READ_TIMEOUT_S,
)
print('概念来源：', CONCEPT_AUDIT_SOURCE or SCREENING_SOURCE)
print('固定图文来源：', ARTICLE_SOURCE, VISUAL_SOURCE)
print('本次试跑：', SAMPLE_SIZE, '个概念；', MODEL, '/', REASONING_EFFORT, '；目标：', OUTPUT_TABLE_URI)
state = await asyncio.to_thread(pipeline.run_pipeline, CONFIG)
display(state)
print('本批实际输入名单（选样发生在正式 pipeline）：')
display(pd.DataFrame(data.read_lance(**state['inputs']).take_all())[['concept', 'taxonomy', 'status']])
print('运行结束后执行第二格：查看全部 5 个 case，含零题与失败原因。')


[T2I V2][glm53_authoring_20260926_02][+0.0s] 开始运行，等待运行锁；概念数=1，模型=glm/glm-5.3-flash，模式=modelhub，每概念一次、串行；续跑先查调用日志
[T2I V2][glm53_authoring_20260926_02][+0.1s] 已取得运行锁，检查并冻结配置及来源版本


ValueError: Run configuration, sources or code changed; use a new run name

In [ ]:
# 第二格：独立只读查看这批 5 个 case；新 kernel 也能直接运行，不再次调用 Codex。
import base64
import html
import json
import sys
from pathlib import Path

import lance
import pandas as pd
from IPython.display import HTML, display

VIEW_PROJECT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
DATA_ROOT = VIEW_PROJECT.parent
for path in (VIEW_PROJECT, DATA_ROOT / 'demiflow'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from demiflow import data
from demiflow.operator_llm.call_ref import read_call
from benchmark.t2i.v2.operaters.concept_context import render_concept_context, ConceptEvidenceError
from preparation.articles.operaters.inputs import pixels

VIEW_RUN_ID = 'dual_keep_codex5_criteria_v3_20260928'  # 与第一格 RUN_ID 一致；查看历史批次时改此处。
CASE_CONCEPT = ''  # 默认全部 5 个；填写准确名称只看一个，不影响上面的出题范围。
VIEW_DIR = VIEW_PROJECT / 'benchmark/t2i/v2/datasets'
RECORDS_URI = str(VIEW_DIR / f'records__{VIEW_RUN_ID}.lance')
from benchmark.t2i.v2.operaters.run_tables import RunTables
view_state = RunTables(DATA_ROOT, RECORDS_URI).load()
# 完成后按 state 的已提交版本读取；尚无 state 时只看已提交输入/设计，明确标为运行中快照。
view_refs = {}
for name in ('inputs', 'designs', 'candidates'):
    if view_state is not None and name in view_state:
        view_refs[name] = view_state[name]
    else:
        uri = str(VIEW_DIR / f'{name}__{VIEW_RUN_ID}.lance')
        if Path(uri).exists():
            view_refs[name] = {'uri': uri, 'version': lance.dataset(uri).version}
print('查看运行：', VIEW_RUN_ID)
print('状态：', {'complete': view_state['complete'], 'counts': view_state.get('counts', {})} if view_state else '尚无完成记录；仅查看已提交快照')
if view_state and view_state.get('sources'):
    print('本批固定图文来源：', view_state['sources'])
if view_state and view_state.get('selection'):
    print('本批粗筛来源与选样：', view_state['selection'])
display(pd.DataFrame([{'表': name, **ref} for name, ref in view_refs.items()]))

# 4. 只读本次 writer 已提交的输入/设计/候选；不只看成功题，确保零题和失败 case 仍显示。
input_rows = data.read_lance(**view_refs['inputs']).take_all() if 'inputs' in view_refs else []
designs = data.read_lance(**view_refs['designs']).take_all() if 'designs' in view_refs else []
questions = data.read_lance(**view_refs['candidates']).take_all() if 'candidates' in view_refs else []
design_by_concept = {row['concept']: row for row in designs}
case_rows = [{**row, **design_by_concept.get(row['concept'], {'status': 'waiting', 'reason': '设计结果尚未落表'})}
             for row in input_rows]
case_summary = pd.DataFrame([{'concept': row['concept'], 'status': row['status'], 'reason': row['reason'],
                             'taxonomy': '\n'.join(row.get('taxonomy') or []),
                             'text_count': sum(ref['kind'] == 'text' for ref in json.loads(row['references_json'])),
                             'image_count': sum(ref['kind'] == 'image' for ref in json.loads(row['references_json'])),
                             'candidate_count': sum(q['concept'] == row['concept'] for q in questions)}
                            for row in case_rows], columns=['concept', 'taxonomy', 'text_count', 'image_count', 'status', 'reason', 'candidate_count'])
print(f'本批输入 {len(input_rows)} 个；已保存设计 {len(designs)} 个；待审候选 {len(questions)} 道。')
display(HTML(case_summary.to_html(index=False, max_rows=None, max_cols=None, escape=True)))
if not case_rows:
    print('本轮尚无输入，请先手工运行第一格；本格不会提交模型请求。')

# 题面、全部考点/依据/判据及无题原因完整显示；调用元信息用于核对独立 Codex 执行及缓存复用。
display(HTML('''<style>
table.t2i-preview {width:100%; table-layout:fixed;}
table.t2i-preview th, table.t2i-preview td {text-align:left; vertical-align:top;
    white-space:pre-wrap; overflow-wrap:anywhere; max-width:none;}
</style>'''))
for index, design in enumerate(case_rows, 1):
    if CASE_CONCEPT and design['concept'] != CASE_CONCEPT:
        continue
    display(HTML(f'<h3>Case {index}/{len(case_rows)}：' + html.escape(design['concept']) + '</h3>'))
    display(HTML('<p><b>完整 taxonomy</b></p><div style="white-space:pre-wrap">'
                 + html.escape('\n'.join(design.get('taxonomy') or []) or '本版本未提供分类') + '</div>'))
    display(HTML(pd.DataFrame([{k: design[k] for k in ('status', 'reason')}]).to_html(
        index=False, max_rows=None, max_cols=None, escape=True, classes='t2i-preview')))
    for question in [q for q in questions if q['concept'] == design['concept']]:
        display(HTML('<p><b>题目（待审）</b>：' + html.escape(question['instruction']) + '</p>'))
        points = pd.DataFrame(question['test_points']).reindex(columns=['point', 'basis', 'criterion'])
        points['criterion'] = points['criterion'].fillna('本历史版本未输出判据')
        display(HTML(points.rename(columns={'point': '考点', 'basis': '依据', 'criterion': '判据（正确标准）'}).to_html(
            index=False, max_rows=None, max_cols=None, escape=True, classes='t2i-preview')))
    if design['status'] != 'candidate':
        display(HTML('<p>本 case 暂无候选题；请查看上方状态和原因。</p>'))
    concept_record = design.get('concept_record')
    if concept_record:
        display(HTML('<details><summary>结构化审定记录与固定来源（点击展开）</summary><div style="white-space:pre-wrap">'
                     + html.escape(json.dumps(concept_record, ensure_ascii=False, indent=2)) + '</div></details>'))
        if concept_record.get('evidence'):
            try:
                context_preview = render_concept_context(concept_record)
            except ConceptEvidenceError as error:
                context_preview = '审定证据读取失败：' + str(error)
        else:
            context_preview = '本记录未交付审定证据映射；不能据引用编号声称已提供原文。'
        display(HTML('<details><summary>按固定引用读取的审定资料（当前排版，点击展开）</summary><div style="white-space:pre-wrap">'
                     + html.escape(context_preview) + '</div></details>'))
    call = json.loads(design.get('call_json') or '{}')
    authoring_context = json.loads(design.get('authoring_context_json') or '[]')
    if authoring_context:
        display(HTML('<details><summary>作者补读原文与算子回执（点击展开，不属于作答 references）</summary><div style="white-space:pre-wrap">'
                     + html.escape(json.dumps(authoring_context, ensure_ascii=False, indent=2)) + '</div></details>'))
    reasoning = design.get('reasoning')
    if reasoning:
        display(HTML('<details><summary>模型返回的 reasoning（点击展开）</summary><div style="white-space:pre-wrap">'
                     + html.escape(reasoning) + '</div></details>'))
    if call.get('response_ref'):
        response = read_call(call['response_ref'], DATA_ROOT)
        body = response.get('body') or {}
        choice = (body.get('choices') or [{}])[0] if isinstance(body, dict) else {}
        display(HTML(pd.DataFrame([{
            'model': call.get('model'), 'reasoning_effort': call.get('reasoning_effort'),
            'transport': call.get('transport', 'http'), 'reused': call.get('reused'),
            'elapsed_s': response.get('elapsed_s'), 'execution_status': call.get('execution_status'),
            'exit_code': call.get('exit_code'), 'finish_reason': choice.get('finish_reason'),
            'usage': json.dumps(call.get('usage') or {}, ensure_ascii=False),
        }]).to_html(index=False, escape=True, classes='t2i-preview')))
        display(HTML('<details><summary>完整原始响应与调用引用（含 CLI 日志，点击展开）</summary>'
                     + '<div style="white-space:pre-wrap">' + html.escape(json.dumps(
                         {'call': call, 'response': response}, ensure_ascii=False, indent=2)) + '</div></details>'))
    references = json.loads(design['references_json'])
    if not references:
        display(HTML('<p>本 case 未提供作答参考图文；审定资料仅供作者核验，实际使用的依据应写在 basis 中。</p>'))
    for ref in references:
        display(HTML('<p>材料 ' + str(ref['number']) + '</p>'))
        if ref['kind'] == 'text':
            display(HTML('<div style="white-space:pre-wrap">' + html.escape(ref['text']) + '</div>'))
        else:
            raw, mime, _ = pixels({'object_ref': ref['object_ref'], 'sha256': ref['object_ref']['sha256']})
            url = 'data:' + mime + ';base64,' + base64.b64encode(raw).decode()
            display(HTML('<details><summary>参考图片（点击展开）</summary><img src="' + url
                         + '" style="max-width:720px;max-height:720px"></details>'))
